<a href="https://colab.research.google.com/github/Kaviyarasi-Sasiperumal/AI_-Based_-Document-_Search_-and-_Knowledge-_Retrieval_-with-_Conversational_Interface/blob/main/Milestone_4_Deployment_%26__Final_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.6 MB/s eta 0:00:00


In [2]:
import gradio as gr
import PyPDF2
import time

In [3]:
document_text = ""
chunks = []

In [4]:
def create_chunks(text, chunk_size=300):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [26]:
def get_answer_from_document(question):
    lines = [l.strip() for l in document_text.split("\n") if l.strip()]
    q = question.lower().strip()

    resume_fields = {
        "degree": ["degree"],
        "branch": ["branch"],
        "specialization": ["specialization"],
        "qualification": ["qualification"],
        "name": ["name"],
        "email": ["email"],
        "phone": ["phone", "mobile", "contact"],
        "skills": ["skills", "technical skills"],
        "objective": ["objective", "career objective"],
        "projects": ["projects"]
    }

    section_headers = list(resume_fields.keys()) + [
        "education", "experience", "certifications",
        "internship", "achievements", "publications",
        "abstract", "introduction",
        "working of artificial intelligence",
        "applications and future scope"
    ]

    for field, keywords in resume_fields.items():
        if any(word in q for word in keywords):

            for i, line in enumerate(lines):
                line_lower = line.lower()


                if any(word in line_lower for word in keywords) and ":" in line:
                    return line.split(":", 1)[1].strip()


                if line_lower in keywords:

                    collected = []
                    j = i + 1

                    while j < len(lines):
                        if lines[j].lower() in section_headers:
                            break
                        collected.append(lines[j])
                        j += 1

                    return "\n".join(collected) if collected else "Not found."

            return "This document does not contain that information."


    if "what is ai" in q or "define ai" in q or "what is artificial intelligence" in q:
        for i, line in enumerate(lines):
            if "artificial intelligence refers to" in line.lower():

                definition = line
                j = i + 1

                while j < len(lines) and not definition.endswith("."):
                    definition += " " + lines[j]
                    j += 1

                return definition.strip()

    for i, line in enumerate(lines):
        if q == line.lower():

            content = []
            j = i + 1

            while j < len(lines) and lines[j].lower() not in section_headers:
                content.append(lines[j])
                j += 1

            return "\n".join(content) if content else "Section found but no content."

    for line in lines:
        if q in line.lower():
            return line

    return "Answer not found in document."

In [21]:
def load_document(files):
    global document_text, chunks
    document_text = ""
    chunks = []

    if not files:
        return "❌ No documents uploaded", ""

    total_pages = 0

    for file in files:
        text = ""

        # PDF
        if file.name.lower().endswith(".pdf"):
            reader = PyPDF2.PdfReader(file)
            total_pages += len(reader.pages)

            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        # TXT
        elif file.name.lower().endswith(".txt"):
            text += file.read().decode("utf-8")
            total_pages += 1

        document_text += text + "\n"

    chunks = create_chunks(document_text)

    stats = f"""
📊 DOCUMENT STATISTICS

📄 Total Documents : {len(files)}
📄 Total Pages     : {total_pages}
🧩 Total Chunks    : {len(chunks)}
📝 Total Words     : {len(document_text.split())}
🔡 Total Characters: {len(document_text)}
"""

    return "✅ Documents processed successfully", stats

In [22]:
chat_history_data = []

def download_chat():
    if not chat_history_data:
        return None

    filename = "chat_history.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.writelines(chat_history_data)

    return filename

In [8]:
def handle_message(user_input, history):

    global chat_history_data

    if history is None:
        history = []

    if not user_input:
        return history, ""

    start = time.perf_counter()

    if user_input.lower() in ["hi", "hello", "hey"]:
        reply = "Hello 👋 How can I help you?"
    elif not document_text:
        reply = "📄 Please upload documents first."
    else:
        answer = get_answer_from_document(user_input)
        response_time = round((time.perf_counter() - start) * 1000, 2)
        reply = f"{answer}\n\n⏱ Response Time: {response_time} ms"

    history = history + [
        {"role": "user", "content": user_input},
        {"role": "assistant", "content": reply},
    ]

    chat_history_data.append(f"User: {user_input}\nBot: {reply}\n\n")

    return history, ""

In [9]:
def clear_chat():
    return []

In [28]:
# -----------------------
# UI
# -----------------------
with gr.Blocks() as demo:

    gr.Markdown("## 🤖 Multi-Document Chatbot")

    with gr.Row():


        with gr.Column(scale=1):

            gr.Markdown("### 📂 Upload PDF / TXT")
            file_input = gr.File(file_count="multiple")

            status = gr.Textbox(label="Status", interactive=False)
            stats_box = gr.Textbox(label="Document Statistics", lines=8, interactive=False)

            file_input.change(load_document, file_input, [status, stats_box])

            clear_btn = gr.Button("🗑 Clear Chat")
            download_btn = gr.Button("⬇ Download Chat")
            download_file = gr.File()


        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                height=450,
                type="messages"
            )

            user_input = gr.Textbox(placeholder="Type your message...")
            send_btn = gr.Button("Send")

            send_btn.click(
                handle_message,
                inputs=[user_input, chatbot],
                outputs=[chatbot, user_input]
            )

            user_input.submit(
                handle_message,
                inputs=[user_input, chatbot],
                outputs=[chatbot, user_input]
            )

    clear_btn.click(clear_chat, outputs=chatbot)
    download_btn.click(download_chat, outputs=download_file)

demo.launch()

/tmp/ipython-input-4145011914.py:28: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db522cfe3a089dbcf1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
